# Lesson 10 — Bayesian optimisation over a surrogate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karefyllidis/PlugFlowML/blob/main/notebooks/Main_10_optimisation_BO_surrogate_vs_cantera.ipynb)

*Part of the [PlugFlowML course](../README.md): machine-learning surrogates for physical simulation, taught on a chemical reactor.*

**Mode:** hands-on (Colab-ready) · **Runtime:** ~10 min on free Colab · **Builds on:** Lesson 8

**What you'll learn**

1. Frame a design question — "which inlet conditions maximise the product yield?" — as a bounded optimisation problem over a trained surrogate.
2. Run Bayesian optimisation with Optuna's Gaussian-process sampler and read its trial history.
3. Explain why optimisers exploit surrogate error ("surrogate optimism") and why the winning point must always be re-checked with the true model.
4. Constrain a search space to the training domain so the surrogate is never asked to extrapolate.
5. Step back over the whole course: the simulate → learn → distil → optimise → validate loop you have now built end to end, and how to transplant it to your own field.

**The example system.** The data comes from a plug-flow reactor (PFR): a hydrocarbon feed flows down a heated tube and progressively cracks into lighter products such as olefins ([Lesson 1](Main_1_run_pfr.ipynb) tells the full story). Six inlet/design knobs — temperature, pressure, tube length and diameter, mass flow rate, wall heat flux — set the product distribution at the exit. Here we hand those six knobs to an optimiser, with the symbolic-regression surrogate from [Lesson 8](Main_8_symbolic_regression_SR.ipynb) standing in for the reactor. The bootstrap cell downloads pretrained SR equations, so you do not need to have run Lesson 8 yourself; the final Cantera validation additionally needs a kinetic mechanism we cannot redistribute, and degrades gracefully to read-along without it.

In [ ]:
# ══ 0. COLAB BOOTSTRAP ══
# Running locally? This cell is a no-op — just run it and move on.
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if Path.cwd().name != "notebooks":            # fresh Colab VM
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/karefyllidis/PlugFlowML.git"], check=True)
        %cd PlugFlowML/notebooks
    %pip install -q optuna cantera
    _REL = "https://github.com/karefyllidis/PlugFlowML/releases/download/sample-data-v1"
    import urllib.request, zipfile
    for _f, _d in [("features_targets_training_data_complete_sample150.pkl", "../data/processed"),
                   ("models_pretrained_sample.zip", "../models")]:
        _p = Path(_d); _p.mkdir(parents=True, exist_ok=True)
        if not (_p / _f).exists():
            print(f"Downloading {_f} …")
            urllib.request.urlretrieve(f"{_REL}/{_f}", _p / _f)
            if _f.endswith(".zip"):
                with zipfile.ZipFile(_p / _f) as _z:
                    _z.extractall(_p)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP
# ════════════════════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path
import importlib.util
import json
import warnings
warnings.filterwarnings('ignore')

# Resolve project root (works whether cwd is repo root or notebooks/)
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
os.chdir(project_root)

# IMPORTANT: import cantera BEFORE adding src/ to sys.path to avoid shadowing
import cantera as ct
print(f'Cantera {ct.__version__}')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.utils.plot_style import setup_matplotlib
from src.ml.data_generation import TrainingDataGenerator

print(f'Optuna {optuna.__version__}')
print(f'Project root: {project_root}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
# Edit configs/ml/main10_bayesian_optimisation_config.json to change these
# (edit and re-run this cell; no kernel restart needed).
CONFIG_PATH = project_root / 'configs' / 'ml' / 'main10_bayesian_optimisation_config.json'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
else:
    _cfg = {}
    print(f'[WARN] Config not found: {CONFIG_PATH}. Using defaults.')

# --- Optimisation target ---
# Column name in the SR target_cols to maximise at reactor exit.
OPT_TARGET   = _cfg.get('opt_target', 'Y_lump_olefins')

# --- Optuna GP budget ---
N_TRIALS     = _cfg.get('n_trials', 150)        # SR surrogate study
RANDOM_STATE = _cfg.get('random_state', 42)

# --- Inlet condition search bounds (training-data domain) ---
_default_bounds = {
    'T_K':    (800.0,    900.0),     # inlet temperature [K]
    'P_Pa':   (1.5e5,    3.5e5),     # inlet pressure [Pa]
    'L_m':    (10.0,     15.0),      # reactor length [m]
    'D_m':    (0.025,    0.040),     # reactor diameter [m]
    'mdot':   (0.05,     0.10),      # mass flow rate [kg/s]
    'q_Wm2':  (1.0e5,    2.5e5),    # wall heat flux [W/m²]
}
BOUNDS = {k: tuple(v) for k, v in _cfg.get('bounds', {}).items()} or _default_bounds

# --- Cantera reactant ---
REACTANT_KEY = _cfg.get('reactant_key', 'n-hexane')

# --- Artefact stems ---
SR_DIR = project_root / 'models' / 'sr_full_profile'   # from Main_8 (TEACHER_STEM = 'simple_nn_full_profile')

# --- Flags ---
IF_PLOT_SHOWN  = _cfg.get('if_plot_shown', True)
IF_PLOT_EXPORT = _cfg.get('if_plot_export', True)
FIGS_DIR = project_root / 'outputs' / 'figures' / 'Main_10_optimisation'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Target:     {OPT_TARGET}')
print(f'N_TRIALS:   {N_TRIALS}')
print(f'Reactant:   {REACTANT_KEY}')

> 🧠 **Concept — Bounds are the training domain.** The search bounds above are not arbitrary: they match the parameter ranges the training data was generated over in Lesson 2's design of experiments. Inside that box the surrogate interpolates between things it has seen; outside it extrapolates, and it does so *silently* — a fitted expression or network returns a number for any input, however far from the data, with no warning attached. An optimiser is the worst possible client for that failure mode: it actively hunts for extreme values, so an unbounded search will happily chase a fictitious peak just beyond the data and report it with full confidence. Constraining the search space to the training domain is therefore not a convenience, it is a correctness requirement. And if the optimum lands *on* a bound, read that as information: the true optimum may lie outside the current domain — a reason to generate more simulation data there, not to quietly widen the bounds.

> 🧠 **Concept — Optimise the surrogate, not the simulator.** This lesson is the payoff of the whole pipeline. One Cantera PFR simulation with a detailed mechanism takes seconds to minutes; the symbolic expression distilled in Lesson 8 evaluates in microseconds — roughly six orders of magnitude cheaper per call. Optimisers need many objective evaluations, so that ratio decides what is feasible: 60 trials against the simulator is a coffee-break-to-overnight job, while against the surrogate it is instant, and you can afford restarts, alternative objectives, tighter bounds and full sensitivity maps for free. The price is subtle but real — you are optimising the surrogate's *opinion* of the reactor, not the reactor — which is why this notebook ends by re-simulating the winner with Cantera and reporting the gap.
>
> *In your domain:* the same pattern powers aerodynamic shape optimisation over CFD surrogates, battery-design searches over electrochemical models, and materials screening over DFT surrogates — anywhere the true model costs minutes to hours per evaluation.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD SR ARTEFACTS (Main_8 symbolic equations)
# ════════════════════════════════════════════════════════════════════════════
sr_eq_path = SR_DIR / 'sr_full_profile_equations.py'
if not sr_eq_path.exists():
    raise FileNotFoundError(
        f'{sr_eq_path} not found — run Main_8 with IF_SR_EXPORT=True first.'
    )

spec = importlib.util.spec_from_file_location('sr_full_profile_equations', sr_eq_path)
sr_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sr_module)

# Build the target function name from OPT_TARGET (mirrors Main_8 export logic)
sr_fn_name = 'predict_' + OPT_TARGET.replace('.', '_').replace(' ', '_')
if not hasattr(sr_module, sr_fn_name):
    available = [x for x in dir(sr_module) if x.startswith('predict_')]
    raise AttributeError(
        f'{sr_fn_name} not found in SR module. Available: {available}'
    )
sr_fn = getattr(sr_module, sr_fn_name)

# Load SR manifest for metadata
sr_manifest_path = SR_DIR / 'sr_full_profile_manifest.json'
sr_meta = json.loads(sr_manifest_path.read_text()) if sr_manifest_path.exists() else {}
sr_inlet_cols = sr_meta.get('inlet_cols', [
    'initial_temperature_K', 'initial_pressure_Pa', 'reactor_length_m',
    'reactor_diameter_m', 'mass_flow_rate_kgps', 'heat_flux_Wm2',
    'z_position_m', 'relative_position',
])

print(f'SR function loaded: {sr_fn_name}')
print(f'SR inlet cols: {sr_inlet_cols}')


def predict_sr(T_K, P_Pa, L_m, D_m, mdot, q_Wm2) -> float:
    """Return SR expression prediction for OPT_TARGET at the reactor exit (z/L = 1)."""
    arg_map = {
        'initial_temperature_K': T_K,
        'initial_pressure_Pa':   P_Pa,
        'reactor_length_m':      L_m,
        'reactor_diameter_m':    D_m,
        'mass_flow_rate_kgps':   mdot,
        'heat_flux_Wm2':         q_Wm2,
        'z_position_m':          L_m,
        'relative_position':     1.0,
    }
    args = [arg_map.get(c, 0.0) for c in sr_inlet_cols]
    try:
        val = float(sr_fn(*args))
    except Exception:
        val = float('nan')
    return val

> 🧠 **Concept — Bayesian optimisation.** When each objective evaluation is expensive, grid and random search waste most of their budget. Bayesian optimisation (BO) instead treats the objective as an unknown function and fits a probabilistic model of it — here a Gaussian process (GP), which returns both a predicted value and an uncertainty for any candidate point. An *acquisition function* then scores candidates by combining the two, trading off *exploitation* (sample where the prediction is high) against *exploration* (sample where the uncertainty is high), and the best-scoring candidate becomes the next trial. Each observation updates the GP posterior, so the search concentrates evaluations where they are most informative; Optuna's `GPSampler` packages the whole loop behind the `suggest_float` calls below. One honest caveat: our objective (the SR expression) is so cheap that brute force would also work here — but the workflow you are about to run is exactly the one you would use if each trial were a real experiment or an hour-long simulation, which is where BO earns its keep.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. OPTUNA GP STUDY — SR SURROGATE
# ════════════════════════════════════════════════════════════════════════════

def sr_objective(trial: optuna.Trial) -> float:
    T_K   = trial.suggest_float('T_K',   *BOUNDS['T_K'])
    P_Pa  = trial.suggest_float('P_Pa',  *BOUNDS['P_Pa'])
    L_m   = trial.suggest_float('L_m',   *BOUNDS['L_m'])
    D_m   = trial.suggest_float('D_m',   *BOUNDS['D_m'])
    mdot  = trial.suggest_float('mdot',  *BOUNDS['mdot'])
    q_Wm2 = trial.suggest_float('q_Wm2', *BOUNDS['q_Wm2'])
    val   = predict_sr(T_K, P_Pa, L_m, D_m, mdot, q_Wm2)
    return float(val) if np.isfinite(val) else -1e9


study_sr = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.GPSampler(seed=RANDOM_STATE),
    study_name='sr_bo',
)
study_sr.optimize(sr_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_sr  = study_sr.best_params
best_sr_val = study_sr.best_value
print(f'\nSR best {OPT_TARGET} = {best_sr_val:.6f}')
for k, v in best_sr.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. OPTIMISATION HISTORY PLOT
# ════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 4.5))
setup_matplotlib([ax])

trials = study_sr.trials
vals   = [t.value for t in trials]
best_so_far = np.maximum.accumulate(vals)

ax.scatter(range(len(vals)), vals, s=6, alpha=0.5, c='r', label='trial value')
ax.plot(range(len(best_so_far)), best_so_far, color='r', lw=1.5, label='best so far')
ax.set_xlabel('Trial')
ax.set_ylabel(OPT_TARGET)
ax.set_title('GP-BO history — SR surrogate')
ax.legend(fontsize=8)

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'bo_history.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

#### 💪 Exercise 10.1 — Find the plateau

The "best so far" curve above flattens well before the last trial. Compute the first trial index at which the study got within 1% of its final best value, then tighten the tolerance to 0.1% and compare. You should see the 1% mark hit within roughly the first half of the budget — evidence that the GP located the promising region quickly and spent the remaining trials refining and exploring. Then decide: would you cut `n_trials` in the config, and what would you be risking?

In [ ]:
# 💪 Exercise 10.1 — your turn (edit and re-run)
# TODO: change TOL to 0.001 (0.1%) and compare the two trial indices.
TOL = 0.01   # within 1% of the final best

vals_ex1 = np.array([t.value for t in study_sr.trials])
best_curve_ex1 = np.maximum.accumulate(vals_ex1)
final_best_ex1 = best_curve_ex1[-1]
threshold_ex1 = final_best_ex1 - TOL * abs(final_best_ex1)
first_hit_ex1 = int(np.argmax(best_curve_ex1 >= threshold_ex1))

print(f'Final best {OPT_TARGET}: {final_best_ex1:.6f}')
print(f'First trial within {TOL:.1%} of the final best: '
      f'trial {first_hit_ex1} of {len(vals_ex1)}')

<details><summary><b>💡 Solution & discussion — Exercise 10.1</b></summary>

```python
TOL = 0.001   # 0.1% — the only line to change

vals_ex1 = np.array([t.value for t in study_sr.trials])
best_curve_ex1 = np.maximum.accumulate(vals_ex1)
final_best_ex1 = best_curve_ex1[-1]
threshold_ex1 = final_best_ex1 - TOL * abs(final_best_ex1)
first_hit_ex1 = int(np.argmax(best_curve_ex1 >= threshold_ex1))
print(f'First trial within {TOL:.2%}: trial {first_hit_ex1} of {len(vals_ex1)}')
```

Typically the 1% threshold is crossed within the first half of the budget while the 0.1% threshold arrives noticeably later — the classic diminishing-returns shape of BO. Whether to cut `n_trials` depends entirely on evaluation cost: against this microsecond surrogate the extra trials are free, so keep them; if each trial were a real experiment you would cap the budget or use a convergence-based stopping rule. One caution: a plateau on the *surrogate* proves nothing about the *true* objective — the flat tail could sit on a region the surrogate gets wrong, which is exactly why the Cantera validation at the end of this lesson exists.
</details>

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. BEST PARAMETERS TABLE
# ════════════════════════════════════════════════════════════════════════════
param_labels = {
    'T_K':   'T_in [K]',
    'P_Pa':  'P_in [bar]',
    'L_m':   'L [m]',
    'D_m':   'D [m]',
    'mdot':  'ṁ [kg/s]',
    'q_Wm2': 'q [kW/m²]',
}
scale = {
    'T_K':   1.0,
    'P_Pa':  1e-5,     # → bar
    'L_m':   1.0,
    'D_m':   1.0,
    'mdot':  1.0,
    'q_Wm2': 1e-3,     # → kW/m²
}

rows = []
for k, label in param_labels.items():
    rows.append({'Parameter': label, 'SR optimal': f'{best_sr[k] * scale[k]:.4g}'})
rows.append({'Parameter': f'{OPT_TARGET} (surrogate)', 'SR optimal': f'{best_sr_val:.6f}'})
df_best = pd.DataFrame(rows)
print(df_best.to_string(index=False))

#### 💪 Exercise 10.2 — Optimise a different product

Steam cracking produces hydrogen alongside olefins, and the SR export from Lesson 8 contains one equation per lumped target — so switching objectives costs nothing. Re-run the optimisation with the hydrogen lump as the objective, in a *fresh* Optuna study, and compare the optimal inlet conditions with the olefins optimum from Section 6. You should see the hydrogen optimum push toward harsher conditions (typically higher inlet temperature and/or heat flux): hydrogen keeps being released as molecules crack again and again, while olefins are intermediates that are themselves destroyed if cracking goes too deep.

In [ ]:
# 💪 Exercise 10.2 — your turn (edit and re-run)
# TODO: swap EX2_TARGET_HINT for another lump from the printed list
#       (e.g. 'paraffins' or 'aromatics') and compare again.
EX2_TARGET_HINT = 'hydrogen'
EX2_N_TRIALS = 40                      # smaller budget — plenty for a look

lump_fns = sorted(n for n in dir(sr_module) if n.startswith('predict_Y_lump'))
print(f'Available lump targets: {lump_fns}')
ex2_fn_name = next((n for n in lump_fns if EX2_TARGET_HINT in n), lump_fns[0])
ex2_fn = getattr(sr_module, ex2_fn_name)
print(f'Optimising: {ex2_fn_name}')


def predict_sr_ex2(T_K, P_Pa, L_m, D_m, mdot, q_Wm2):
    """Exit-plane (z/L = 1) prediction of the exercise target."""
    arg_map = {'initial_temperature_K': T_K, 'initial_pressure_Pa': P_Pa,
               'reactor_length_m': L_m, 'reactor_diameter_m': D_m,
               'mass_flow_rate_kgps': mdot, 'heat_flux_Wm2': q_Wm2,
               'z_position_m': L_m, 'relative_position': 1.0}
    try:
        return float(ex2_fn(*[arg_map.get(c, 0.0) for c in sr_inlet_cols]))
    except Exception:
        return float('nan')


def ex2_objective(trial):
    p = {k: trial.suggest_float(k, *BOUNDS[k]) for k in BOUNDS}
    v = predict_sr_ex2(**p)
    return v if np.isfinite(v) else -1e9


study_ex2 = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.GPSampler(seed=RANDOM_STATE),
                                study_name='ex_10_2')
study_ex2.optimize(ex2_objective, n_trials=EX2_N_TRIALS, show_progress_bar=False)

print(f'\nBest {ex2_fn_name.replace("predict_", "")} = {study_ex2.best_value:.6f}')
df_ex2 = pd.DataFrame({'olefins optimum (Section 4)': best_sr,
                       f'{EX2_TARGET_HINT} optimum': study_ex2.best_params})
print(df_ex2.round(4).to_string())

<details><summary><b>💡 Solution & discussion — Exercise 10.2</b></summary>

```python
EX2_TARGET_HINT = 'paraffins'   # the only line to change (or 'aromatics', …)
# ... rest of the scaffold unchanged ...
```

The scaffold already runs the hydrogen case as shipped. Comparing the two columns of the printed table, the hydrogen optimum typically sits at (or very near) the upper temperature and heat-flux bounds, because every further cracking step releases more H₂; the olefins optimum is milder, since ethylene and propylene are intermediates that over-crack into aromatics and coke precursors if pushed too hard. Two methodological points carry beyond chemistry: (1) with a per-target SR export, changing the objective is a one-line edit and a few seconds of compute — try doing that with a simulator-in-the-loop optimisation; (2) both optima leaning on bounds echoes the "bounds are the training domain" concept box — the surrogate cannot tell you what lies beyond, only new simulations can.
</details>

#### 💪 Exercise 10.3 — Tighten a bound

Suppose the furnace cannot deliver more than 850 K at the inlet. Cap the temperature bound at 850 K in a *copy* of the bounds dict, re-optimise in a fresh study, and compare both the optimal conditions and the achievable objective with Section 4. You should see the optimiser pin temperature at the new upper bound and partially compensate with the other knobs (typically more wall heat flux and/or longer residence time), with the best value dropping only modestly — a useful sensitivity result you got for free from the surrogate.

In [ ]:
# 💪 Exercise 10.3 — your turn (edit and re-run)
# TODO: tighten a different knob instead, e.g. BOUNDS_EX3['q_Wm2'] = (1.0e5, 1.5e5),
#       and see which other parameters take up the slack.
BOUNDS_EX3 = dict(BOUNDS)              # copy — never mutate the pipeline's BOUNDS
BOUNDS_EX3['T_K'] = (BOUNDS['T_K'][0], 850.0)   # furnace limit: inlet T ≤ 850 K


def ex3_objective(trial):
    p = {k: trial.suggest_float(k, *BOUNDS_EX3[k]) for k in BOUNDS_EX3}
    v = predict_sr(**p)
    return v if np.isfinite(v) else -1e9


study_ex3 = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.GPSampler(seed=RANDOM_STATE),
                                study_name='ex_10_3')
study_ex3.optimize(ex3_objective, n_trials=40, show_progress_bar=False)

print(f'Unconstrained best {OPT_TARGET}: {best_sr_val:.6f}  '
      f'(T = {best_sr["T_K"]:.1f} K)')
print(f'Constrained best   {OPT_TARGET}: {study_ex3.best_value:.6f}  '
      f'(T = {study_ex3.best_params["T_K"]:.1f} K)')
print('\nConstrained optimum vs bounds:')
for k in BOUNDS_EX3:
    print(f'  {k:>6}: {study_ex3.best_params[k]:10.4g}   bounds {BOUNDS_EX3[k]}')

<details><summary><b>💡 Solution & discussion — Exercise 10.3</b></summary>

```python
BOUNDS_EX3 = dict(BOUNDS)
BOUNDS_EX3['q_Wm2'] = (1.0e5, 1.5e5)   # alternative: cap the wall heat flux instead
# ... rest of the scaffold unchanged ...
```

With the 850 K cap, the constrained optimum usually sits exactly on the new temperature bound — an *active constraint* — and the optimiser claws back most of the lost yield through the remaining knobs, so the best value drops by less than the naive expectation. Two readings of that result: for the engineer, it quantifies what the furnace limit actually costs; for the modeller, an optimum pinned to a bound is a flag that the interesting region continues beyond it, so if the limit is negotiable the right next step is to generate training data past 850–900 K, retrain, and re-optimise — not to let the surrogate extrapolate.
</details>

#### 💪 Exercise 10.4 — Map the landscape around the optimum

The optimiser reports a single point; a picture tells you whether that point sits on a sharp peak or a broad plateau. Evaluate the SR surrogate on a 2-D grid of inlet temperature vs wall heat flux — the other four knobs frozen at their Section 4 optimal values — and draw the yield landscape. You should see a smooth surface with the starred optimum in the high-value region; from the width of that region you can judge how forgiving the process is to drift in T or q. (1,600 surrogate evaluations take well under a second — the same map via Cantera would be about a day of compute.)

In [ ]:
# 💪 Exercise 10.4 — your turn (edit and re-run)
# TODO: swap the slice axes, e.g. X_KEY, Y_KEY = 'mdot', 'L_m', and re-run.
X_KEY, Y_KEY = 'T_K', 'q_Wm2'
N_GRID = 40

x_grid = np.linspace(*BOUNDS[X_KEY], N_GRID)
y_grid = np.linspace(*BOUNDS[Y_KEY], N_GRID)
Z_grid = np.empty((N_GRID, N_GRID))
frozen = dict(best_sr)                 # copy of the Section 4 optimum
for i, yv in enumerate(y_grid):
    for j, xv in enumerate(x_grid):
        p = dict(frozen)
        p[X_KEY], p[Y_KEY] = xv, yv
        Z_grid[i, j] = predict_sr(**p)

fig, ax = plt.subplots(figsize=(6.5, 5))
setup_matplotlib([ax])
im = ax.pcolormesh(x_grid, y_grid, Z_grid, cmap='magma', shading='auto')
ax.plot(best_sr[X_KEY], best_sr[Y_KEY], marker='*', ls='none', color='lime',
        markersize=14, markeredgecolor='k', label='BO optimum')
ax.set_xlabel(X_KEY)
ax.set_ylabel(Y_KEY)
ax.set_title(f'SR surrogate landscape — {OPT_TARGET} at exit\n'
             f'(other knobs frozen at the optimum)')
fig.colorbar(im, ax=ax, label=OPT_TARGET)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close(fig)

<details><summary><b>💡 Solution & discussion — Exercise 10.4</b></summary>

```python
X_KEY, Y_KEY = 'mdot', 'L_m'   # the only line to change
# ... rest of the scaffold unchanged ...
```

The T–q slice typically shows yield rising toward high temperature and heat flux with a broad high-value region rather than a needle-sharp peak — good news operationally, since modest drift in either knob costs little yield. The mdot–L slice shows the residence-time trade-off instead: long tubes with low flow crack deeply, short tubes with high flow barely react. Two caveats keep the picture honest: this is a 2-D *slice* of a 6-D landscape, not the landscape itself (interactions with the frozen knobs are invisible); and it is the *surrogate's* landscape — smooth by construction for an SR expression — so features near the edges of the data domain deserve the least trust. That is exactly the mindset the next section formalises.
</details>

> 🧠 **Concept — Surrogate optimism (the winner's curse).** An optimiser does not find high values of the objective; it finds high values of *objective + model error*. Wherever the surrogate happens to over-predict, the optimiser is preferentially drawn, so the returned "best value" is biased upward — a statistical effect known as the winner's curse, and it appears even when the surrogate's average error is small and unbiased. The gap is often *larger* than the surrogate's test-set error precisely because the search actively seeks out the flattering regions. The corollary is non-negotiable: always re-evaluate the winning configuration with the true model before believing it, and report the validated value, not the surrogate's claim. That single expensive evaluation is the cheapest insurance in the whole workflow.

## From surrogate claim to ground truth

Everything so far happened inside the surrogate's head. Section 7 takes the winning inlet conditions and runs a **real Cantera PFR simulation** at exactly those settings — the same physics code that generated the training data in Lesson 2. Section 8 then compares two numbers:

- the **surrogate-predicted objective** at the optimum (what BO reported), and
- the **simulated objective** at the same conditions (what the reactor "actually" does).

The difference between them is the **surrogate error at the optimum** — the honest error bar on this whole exercise. Typical outcome for this pipeline: the Cantera value lands a few per cent *below* the surrogate's claim (surrogate optimism at work — see the concept box above), while the optimal conditions themselves are physically sensible: high inlet temperature and heat flux, with geometry and flow balancing residence time against over-cracking. If the gap were large, the fix would not be more BO trials — it would be a better surrogate in that region (more training data, or a stronger teacher model).

> ⚠️ **This part needs the proprietary kinetic mechanism** (`mechanisms/*.yaml`, git-ignored — see Lesson 1). The guard cell below checks whether you have it. Without it, Sections 7–8 become read-along — the expected outcome is exactly the paragraph above — and everything you ran up to here is unaffected.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6b. MECHANISM GUARD — can the Cantera validation run?
# ════════════════════════════════════════════════════════════════════════════
# Sections 7–8 need the kinetic mechanism for REACTANT_KEY. Resolve the exact
# file src/cantera/pfr_simulator.py will load (via
# configs/simulation/main1_reactant_database.json → reactants[*].mechanism_file,
# a path relative to the project root) and check that it exists.
_reactant_db_path = project_root / 'configs' / 'simulation' / 'main1_reactant_database.json'
with open(_reactant_db_path) as f:
    _reactant_db = json.load(f)
if REACTANT_KEY not in _reactant_db['reactants']:
    raise KeyError(f"Unknown reactant_key '{REACTANT_KEY}' — "
                   f"choose one of {list(_reactant_db['reactants'])}")
MECH_PATH = project_root / _reactant_db['reactants'][REACTANT_KEY]['mechanism_file']
MECHANISM_AVAILABLE = MECH_PATH.exists()

if MECHANISM_AVAILABLE:
    print(f'Mechanism found: {MECH_PATH.name}')
    print('Sections 7-8 will run the real Cantera validation.')
else:
    print(f'Mechanism file not found: {MECH_PATH}')
    print('Sections 7-8 need it and will skip gracefully. If you cloned the repo '
          'normally this should already be present — check your checkout.')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. CANTERA VALIDATION
# Run Cantera at the SR optimum and extract exit-plane values.
# Skips gracefully when the kinetic mechanism is unavailable (§6b guard).
# ════════════════════════════════════════════════════════════════════════════
import logging
logging.getLogger('cantera').setLevel(logging.ERROR)

# Species that map to Y_lump_olefins (primary olefin products in steam cracking)
OLEFIN_SPECIES = ['Y_C2H4', 'Y_C3H6', 'Y_C4H6', 'Y_C4H8']


def _run_cantera_and_extract(params: dict, sim_id: int) -> dict:
    """Run a single Cantera PFR and return exit-plane key values."""
    gen = TrainingDataGenerator(output_dir=str(project_root / 'data' / 'training'), disable_plots=True)
    df = gen.run_single_simulation(REACTANT_KEY, params, sim_id=sim_id)
    if df is None or len(df) == 0:
        return {'error': 'Cantera simulation failed'}
    row = df.iloc[-1]   # exit plane
    result = {'T_exit_K': float(row.get('temperature_K', row.get('T', float('nan'))))}
    # Sum olefin mass fractions
    olefin_total = sum(float(row.get(sp, 0.0)) for sp in OLEFIN_SPECIES)
    result['Y_lump_olefins_cantera'] = olefin_total
    # Also capture any direct Y_lump_ columns if present
    for col in df.columns:
        if col.startswith('Y_lump_'):
            result[col + '_cantera'] = float(row[col])
    return result


# Build params dict from Optuna best
def _optuna_to_params(best: dict) -> dict:
    return {
        'temperature_K':      best['T_K'],
        'pressure_Pa':        best['P_Pa'],
        'length_m':           best['L_m'],
        'diameter_m':         best['D_m'],
        'mass_flow_rate_kgps': best['mdot'],
        'heat_flux_Wm2':      best['q_Wm2'],
    }


ct_sr = None
if MECHANISM_AVAILABLE:
    print('Running Cantera at SR-optimal conditions ...')
    ct_sr = _run_cantera_and_extract(_optuna_to_params(best_sr), sim_id=1)
    _val = ct_sr.get('Y_lump_olefins_cantera', ct_sr.get('Y_lump_olefins'))
    if isinstance(_val, float):
        print(f'  Done. Y_lump_olefins (Cantera) = {_val:.6f}')
    else:
        print(f'  Cantera returned no olefins value: {ct_sr}')
else:
    print('Read-along from here: the kinetic mechanism is not redistributable, '
          'so the Cantera validation below is shown with committed reasoning/'
          'expected output described in markdown. If you have your own '
          'mechanism, drop it in mechanisms/ and re-run.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. COMPARISON PLOT — surrogate prediction vs Cantera ground truth
# Runs only when §7 produced a validation result (§6b guard).
# ════════════════════════════════════════════════════════════════════════════

def _get_cantera_olefins(ct_result: dict) -> float:
    if 'Y_lump_olefins_cantera' in ct_result:
        return ct_result['Y_lump_olefins_cantera']
    return ct_result.get('Y_lump_olefins', float('nan'))


if ct_sr is None or 'error' in ct_sr:
    print('Skipping comparison plot — no Cantera validation result available.')
    print('Expected outcome once it runs: two bars, with the Cantera bar a few per')
    print('cent below the surrogate bar; that gap is the surrogate error at the')
    print('optimum. See the markdown above Section 6b for the full interpretation.')
else:
    val_sr_surrogate = best_sr_val
    val_sr_cantera   = _get_cantera_olefins(ct_sr)

    err_sr = abs(val_sr_surrogate - val_sr_cantera) / (abs(val_sr_cantera) + 1e-12) * 100

    # ── Bar chart ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 4.5))
    setup_matplotlib([ax])

    labels  = ['SR (surrogate)', 'SR (Cantera)']
    values  = [val_sr_surrogate, val_sr_cantera]
    hatches = ['', '///']

    bars = ax.bar(labels, values, color='white', edgecolor='r', hatch=hatches, linewidth=1.4)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel(OPT_TARGET)
    ax.set_title(f'Surrogate optimum vs Cantera ground truth — {OPT_TARGET}')
    ax.tick_params(axis='x', labelrotation=15)

    ax.annotate(f'SR error: {err_sr:.1f}%', xy=(0.5, 0.02), xycoords='axes fraction',
                color='r', fontsize=8, ha='center')

    plt.tight_layout()
    if IF_PLOT_EXPORT:
        fig.savefig(FIGS_DIR / 'surrogate_vs_cantera.png', dpi=200, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

    print(f'SR → surrogate: {val_sr_surrogate:.6f}  |  Cantera: {val_sr_cantera:.6f}  |  error: {err_sr:.2f}%')

---

## Summary

| | SR (Main_8) |
|---|---|
| Surrogate type | PySR closed-form expression |
| Optimiser | Optuna `GPSampler` |
| Trials | `N_TRIALS` |
| Objective | max `Y_lump_olefins` at exit |

### Interpretation
- **Surrogate error**: difference between the value the surrogate predicted at its optimum and what Cantera computes at the same conditions. Lower = the surrogate is reliable in this region.
- **SR advantage**: the SR expression is differentiable in closed form, making it suitable for gradient-based optimisation and embedding in process simulators without a PyTorch runtime.

> 🧠 **Concept — Closing the loop.** Look back at what you built across these ten lessons: a physics simulator generated training data (Lessons 1–2); you explored and shaped it into features (3); trained and honestly evaluated surrogates of increasing sophistication — trees, neural networks, physics-informed networks (4–7); distilled a network into portable closed-form equations (8); validated the whole chain against ground truth (9); and finally searched the design space thousands of times faster than the simulator allows, paying one full-price simulation to certify the winner (10). That last simulation also closes the loop: the validated optimum — and any region where the surrogate proved wrong — can be simulated properly, appended to the training set, and the surrogate refit, which is exactly how iterative optimise-validate-retrain workflows run in practice.
>
> *In your domain:* swap Cantera for whatever expensive model you own — a climate ensemble, a CFD solver, a battery cell model, an epidemic simulator — and the loop is unchanged: simulate → learn → distil → optimise → validate.